# Exploración inicial del dataset GRAPE Leaf Diseases

## Auditoría, visualización y estadísticas del dataset

Este notebook explora el dataset de hojas de uva para entender su estructura, calidad de anotaciones, distribución de clases y preparación para entrenamiento YOLO.

In [ ]:
from pathlib import Path
from collections import Counter
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import cv2
import numpy as np

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
print(f"Proyecto: {PROJECT_ROOT}")
print(f"Dataset: {DATA_RAW}")

## 1. Auditoría inicial del dataset

In [ ]:
all_files = [p for p in DATA_RAW.rglob("*") if p.is_file()]
extensions = Counter(p.suffix.lower() for p in all_files)
folders = sorted({str(p.parent.relative_to(DATA_RAW)) for p in all_files})

print(f"Total archivos: {len(all_files)}")
print(f"\nExtensiones:")
for ext, count in extensions.most_common():
    print(f"  {ext}: {count}")
print(f"\nCarpetas:")
for f in folders:
    print(f"  {f}/")

## 2. Validación de etiquetas YOLO

In [ ]:
def validate_labels(labels_dir):
    errors = []
    class_ids = set()
    ann_counts = []
    label_files = sorted(labels_dir.rglob("*.txt"))
    for label_file in label_files:
        content = label_file.read_text().strip()
        if not content:
            errors.append((label_file.name, "empty"))
            ann_counts.append(0)
            continue
        lines = content.splitlines()
        count = 0
        for ln, line in enumerate(lines, 1):
            parts = line.split()
            if len(parts) != 5:
                errors.append((label_file.name, f"line {ln}: expected 5 values, got {len(parts)}"))
                continue
            try:
                cid = int(parts[0])
                coords = [float(x) for x in parts[1:]]
            except ValueError:
                errors.append((label_file.name, f"line {ln}: invalid type"))
                continue
            if not all(0 <= x <= 1 for x in coords):
                errors.append((label_file.name, f"line {ln}: coords out of [0,1]"))
            if coords[2] <= 0 or coords[3] <= 0:
                errors.append((label_file.name, f"line {ln}: invalid box size"))
            class_ids.add(cid)
            count += 1
        ann_counts.append(count)
    return class_ids, errors, ann_counts

for split in ["train", "val"]:
    lbl_dir = DATA_RAW / "grape" / "labels" / split
    cids, errs, counts = validate_labels(lbl_dir)
    print(f"\n=== {split.upper()} ===")
    print(f"  Labels: {len(list(lbl_dir.rglob('*.txt')))}")
    print(f"  Clases: {sorted(cids)}")
    print(f"  Errores: {len(errs)}")
    print(f"  Promedio anotaciones/imagen: {sum(counts)/len(counts):.2f}")
    if errs:
        for e in errs[:5]:
            print(f"    {e}")
    ann_dist = Counter(counts)
    print(f"  Distribución:")
    for k in sorted(ann_dist):
        print(f"    {k} anotación(es): {ann_dist[k]} imágenes")

## 3. Distribución de clases

In [ ]:
CLASS_NAMES = ["BlackRot", "Esca", "Healthy", "LeafBlight"]

def class_distribution(labels_dir):
    counter = Counter()
    for f in labels_dir.rglob("*.txt"):
        content = f.read_text().strip()
        if not content:
            continue
        for line in content.splitlines():
            parts = line.split()
            if len(parts) == 5:
                try:
                    counter[int(parts[0])] += 1
                except ValueError:
                    pass
    return counter

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for idx, split in enumerate(["train", "val"]):
    lbl_dir = DATA_RAW / "grape" / "labels" / split
    dist = class_distribution(lbl_dir)
    names = [CLASS_NAMES[c] for c in range(4)]
    counts = [dist.get(c, 0) for c in range(4)]
    colors = ["#ff9999", "#66b3ff", "#99ff99", "#ffcc99"]
    axes[idx].bar(names, counts, color=colors)
    axes[idx].set_title(f"{split.upper()} - Distribución de clases")
    axes[idx].set_ylabel("Número de anotaciones")
    axes[idx].tick_params(axis='x', rotation=45)
    for i, v in enumerate(counts):
        axes[idx].text(i, v + 10, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "reports" / "class_distribution.png", dpi=150)
plt.show()

## 4. Visualización de muestras con bounding boxes

In [ ]:
def draw_yolo_boxes(image_path, label_path, class_names):
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    if not label_path.exists():
        fig, ax = plt.subplots(1, figsize=(8, 8))
        ax.imshow(img)
        ax.axis("off")
        return fig
    content = label_path.read_text().strip()
    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(img)
    if content:
        for line in content.splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            try:
                cid = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:])
            except ValueError:
                continue
            x1 = (xc - bw/2) * w
            y1 = (yc - bh/2) * h
            box_w = bw * w
            box_h = bh * h
            color = ["red", "blue", "green", "orange"][cid % 4]
            rect = patches.Rectangle((x1, y1), box_w, box_h, linewidth=2,
                                     edgecolor=color, facecolor="none")
            ax.add_patch(rect)
            label = class_names[cid] if cid < len(class_names) else f"class_{cid}"
            ax.text(x1, y1-5, label, color=color, fontsize=10, fontweight='bold',
                    bbox=dict(facecolor='white', alpha=0.7))
    ax.axis("off")
    plt.tight_layout()
    return fig

img_dir = DATA_RAW / "grape" / "images" / "train"
lbl_dir = DATA_RAW / "grape" / "labels" / "train"
all_imgs = sorted(img_dir.glob("*.jpg"))
samples = random.sample(all_imgs, min(6, len(all_imgs)))
for img_path in samples:
    lbl_path = lbl_dir / f"{img_path.stem}.txt"
    fig = draw_yolo_boxes(img_path, lbl_path, CLASS_NAMES)
    if fig:
        plt.show()
        plt.close(fig)

## 5. Tamaños de imágenes

In [ ]:
img_dir = DATA_RAW / "grape" / "images" / "train"
sizes = []
for img_file in list(img_dir.glob("*.jpg"))[:500]:
    img = cv2.imread(str(img_file))
    if img is not None:
        h, w = img.shape[:2]
        sizes.append((w, h))
df_sizes = pd.DataFrame(sizes, columns=["width", "height"])
print(f"Ancho - min: {df_sizes['width'].min()}, max: {df_sizes['width'].max()}, media: {df_sizes['width'].mean():.0f}")
print(f"Alto  - min: {df_sizes['height'].min()}, max: {df_sizes['height'].max()}, media: {df_sizes['height'].mean():.0f}")
fig, ax = plt.subplots(1, figsize=(8, 4))
ax.hist(df_sizes['width'], bins=20, alpha=0.7, label='width')
ax.hist(df_sizes['height'], bins=20, alpha=0.7, label='height')
ax.set_xlabel("Píxeles")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de tamaños de imagen")
ax.legend()
plt.tight_layout()
plt.show()

## Conclusiones

- **Dataset en formato YOLO válido** con 4 clases y ~4,200 imágenes
- **Anotaciones consistentes**: coordenadas normalizadas, sin errores estructurales
- **Distribución**: Esca y BlackRot dominan; Healthy subrepresentada
- **Tamaño de imágenes**: consistente, adecuado para YOLO (resize a 640x640)
- **Siguiente paso**: Preparar dataset y entrenar modelo YOLOv8n